# FFM_3_10 — mesh → latent：耳部 embedding + 9 个测量量 + 头部网格其余部分（只用开发集）

**设计已登记**：PROJECT_LOG §2C "2026-10-23 — FFM_3_10"（写代码前登记）。方案原稿是 `Thesis/notes/FFM_3_10_plan_2026-10-23.md`，和登记不一致的地方以 log 为准（主统计量是全频段 ΔLSD，低频 ΔR² 是次要统计量）。

- 和 FFM_3_8 **逐字相同**的 cell：Cell 2（模型定义）、Cell 3（载入 FFM48k_measim）、Cell 4（回归引擎）、Cell 5（数据、latent、几何、指标）、Cell 6（开发集）。不要改。
- 本实验的 cell：Cell 0（路径）、Cell 1（登记）、Cell 7（复现关卡）、Cell 8（REST 和 HEADM 特征）、Cell 9（各行交叉验证，实测和仿真两个目标）、Cell 10（主检验、C1 的 100 次置换、次要统计量、判读）、Cell 11（保存）。
- **直接 Run all。** 用 GPU 运行时。大部分时间在 Cell 10 的 C1（100 次打乱，每次都重选 w 和 α 并解码），它会打印已用时间和预计剩余时间。

**规则**
- 只用 164 个开发集受试者。锁定的 36 人在 2026-10-12 已经用掉（FFM_3_8）；本 notebook **没有测试 cell**，也不读他们的任何数据。
- 跑之前如果要改设计，先在 PROJECT_LOG 另记一条修订。
- 关卡不通过就停（Cell 7：R0 和 R1 复现 FFM_3_8；Cell 9 开头：缓存的 kernel 和 R5 的代码在 w = 0 时复现 R1；R1 的 0.2–5 kHz R² 复现 Task 1）。


In [ ]:
!pip install healpy -q

In [ ]:
# ==========================================================================
# Cell 0 — 路径 · 数据（zip 按 SHA-256 核对后解压到本地盘；读到的每个数组再按 manifest 核对）
# ==========================================================================
import torch
import os, json, csv, time, hashlib, zipfile, shutil
import numpy as np
import healpy as hp
from scipy import stats
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive'
SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
PATHS = dict(
    data_zip    = f'{DRIVE}/ELEC5021/data_48k_2026-10-10.zip',
    split       = f'{DRIVE}/ELEC5021/split_48k.json',
    ckpt        = f'{SAVE_DIR}/FFM48k_measim_best.pth',
    samples_zip = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples.zip',
    # 下面两个只取受试者名单和折号（与 FFM_3_2_1 / Task 1 同一批人、同一组折）；里面 FFM26 的 latent 和旧预测都不读。
    # 48 kHz 的 latent 在 Cell 6 用 FFM48k_measim 现算
    subject_lists = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',
    fold_ref      = f'{SAVE_DIR}/mesh2z_rff_v1/oof_ridge.npz',
)
for k, p in PATHS.items():
    print(f'{k:14s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放好（路径与 Task 1 相同）'
ZIP_SHA256 = 'f51c6b072123e70d6d8a0691373ab90e3f6db9856ce6306f65d949c87fbf97c7'
SPLIT_SHA256 = 'dc89bc454fef351951283cccb3772fdd3e552bbdde5d13668f8f576f4cd83561'
SAMPLES_SHA256 = '8340bdf73be1a8722015e877bcca2957570293484277989f1245eb8d65aa085a'
OUT = f'{SAVE_DIR}/mesh2z_48k_rest_v1'
os.makedirs(OUT, exist_ok=True)
# 锁定的 36 人只能用一次：与 FFM_3_5 / 3_6 / 3_7 共用标记
TEST_MARKS = [f'{SAVE_DIR}/TEST_SPENT_SONICOM_locked36.json', f'{SAVE_DIR}/geometry2z_tvt_v1/TEST_SPENT_mesh.json',
              f'{SAVE_DIR}/geometry2z_tvt_v2/TEST_SPENT_mesh.json', f'{SAVE_DIR}/mesh2z_48k_v1/TEST_SPENT.json']


def sha256(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''):
            h.update(b)
    return h.hexdigest()


def unzip_checked(zip_path, sha, dest):
    stamp = f'{dest}/.zip_ok_{sha[:16]}'
    if os.path.exists(stamp):
        return
    t0 = time.time()
    assert sha256(zip_path) == sha, f'{zip_path}: SHA-256 不符'
    shutil.rmtree(dest, ignore_errors=True); os.makedirs(dest)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(dest)
    open(stamp, 'w').write(sha)
    print(f'{os.path.basename(zip_path)} 核对并解压 ({time.time() - t0:.0f}s)')


unzip_checked(PATHS['data_zip'], ZIP_SHA256, '/content/data_48k_local')
unzip_checked(PATHS['samples_zip'], SAMPLES_SHA256, '/content/mesh2z_samples_local')
DATA, SAMPLES = '/content/data_48k_local/data_48k', '/content/mesh2z_samples_local/mesh2z_samples'
assert os.path.exists(f'{SAMPLES}/manifest.csv')
assert sha256(PATHS['split']) == SPLIT_SHA256, 'split_48k.json 的 SHA-256 不符'
SPLIT = json.load(open(PATHS['split']))
MIRROR = np.load(f'{DATA}/healpix_mirror_idx.npy')
assert np.array_equal(MIRROR[MIRROR], np.arange(3072))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_grad_enabled(False)
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# FFM_3_10：锁定的 36 人已经用过（FFM_3_8，2026-10-12）。这里只检查标记，不写、不读测试数据
assert os.path.exists(f'{SAVE_DIR}/mesh2z_48k_v1/TEST_SPENT.json') or os.path.exists(TEST_MARKS[0]), \
    '找不到 FFM_3_8 的 TEST_SPENT 标记 —— 先确认锁定测试集的状态'
print('锁定的 36 人：已用（FFM_3_8）。本 notebook 只用开发集。')

In [ ]:
# ==========================================================================
# Cell 1 — 设定与登记（已登记：PROJECT_LOG §2C 2026-10-23，写代码前；改设计要在跑之前另记修订）
# ==========================================================================
# ---- 与 FFM_3_8 Cell 1 相同（Cell 2–6 依赖这些名字）
N_BINS, FS = 86, 48000
FREQS = np.arange(N_BINS) * FS / 256
EVAL_BINS = slice(1, N_BINS)              # bin 1–85 = 187.5 Hz – 15.94 kHz，不含 DC
EL = 90 - np.degrees(hp.pix2ang(16, np.arange(3072))[0])
MEAS_DIRS = EL >= -45.0                   # 2652 个方向
ALPHA_REL = list(np.logspace(-4, 5, 37))
N_FOLDS, N_INNER, SEED = 10, 5, 0         # 与 FFM_3_2_1 / Task 1 / FFM_3_8 相同的折
N_BOOT, BS = 2000, 16
SIGMAS, M_RFF = [0.005, 0.01, 0.02, 0.05, 0.1], 256
DM, DS = 2, 4                             # FFM48k_measim 的 FiLM 行：SONICOM 实测、SONICOM 仿真

# ---- FFM_3_10 新增
EAR_R, Z_CUT = 45.0, -75.0                 # 耳部裁剪半径、脖子截断（与 sample_meshes.py / FFM_3_2_1 相同）
N_REST = 20000                             # REST 每人用的点数（全部 5 万个 head_pts 过滤后取前 N_REST 个）
W_GRID = [0.0, 0.25, 0.5, 1.0, 2.0]        # R5：REST 的来源权重，和 α 一起由内层 CV 选
N_NULL = 100                               # C1：REST 按人打乱的次数（种子 0–99），每次重选 (w, α)
BANDS = {'0.2-5 kHz': (200, 5000), '6-12 kHz': (6000, 12000), '12-16 kHz': (12000, 16000)}   # Task 1 的定义，两端都含
GAP = {'5-6 kHz (no band)': (5000.1, 5999.9)}                                                # bin 27–31：不在 Task 1 的频带里，只描述
BIN_BAND = {b: (FREQS >= lo) & (FREQS <= hi) for b, (lo, hi) in {**BANDS, **GAP}.items()}
assert [np.flatnonzero(m)[[0, -1]].tolist() for m in BIN_BAND.values()] == [[2, 26], [32, 64], [64, 85], [27, 31]]
LOW = '0.2-5 kHz'
RATIO_PRED = 1 / 3.1                       # 低频 ΔR² 实测 / 仿真 的预测值（Task 2：实测低频的人际方差是仿真的 3.1 倍）

REGISTRATION = dict(
    logged='PROJECT_LOG §2C, 2026-10-23, registered before any code',
    question=('On the 164 SONICOM development subjects, does adding the head mesh outside both ear crops (REST) to the '
              'current geometry (nine measures + ear embedding, GEO) improve the prediction of the measured HRTF?'),
    encoder='FFM48k_measim_best.pth (frozen), as FFM_3_8',
    sources=dict(
        GEO='FFM_3_8 geometry_blocks(): nine measures + ear-crop RFF mean embedding at 5 scales (unchanged)',
        REST=('all head_pts with z > -75 mm and farther than 45 mm from both ear canals, first 20000 of those, head frame, '
              'whole head mirrored (y -> -y) for right-ear rows; RFF mean embedding at the 5 scales of GEO'),
        HEADM=('head width (y range at |z| < 20 mm, 0.5-99.5 pct), back of head (-x 0.5 pct), front of head (x 99.5 pct), '
               'convex-hull volume, canal x minus median head x; from head points with z > -75 mm'),
    ),
    weighting='each source unit-scaled and added (FFM_3_8 source_kernel), so every source has the same weight; R5 scales REST by w',
    rows={'R0': 'training mean', 'R1': 'GEO (gate: FFM_3_8 development row)',
          'R5': 'GEO + w*REST, w in W_GRID and alpha chosen jointly by the inner CV of each outer training fold (primary)',
          'R2': 'GEO + REST (w = 1)', 'R3': 'GEO + HEADM', 'R4': 'GEO + REST + HEADM', 'R6': 'REST only', 'R7': 'HEADM only',
          'C1': 'R5 with REST permuted across subjects (both ears together), 100 permutations (seeds 0-99), w and alpha re-chosen'},
    targets='measured latent (primary); simulated latent (SONICOM_SIM, diagnostic, same rows and rules)',
    primary_metric='LSD, bins 1-85 (DC excluded), elevation >= -45 deg; per subject = mean of both ears',
    primary_test=('LSD(R5) - LSD(R1), paired over subjects: bootstrap 95% CI (2000), one-sided paired t; '
                  'C1 null of the same difference, p_C1 = (1 + #{C1 <= observed}) / 101'),
    reading=("the head beyond the ear crops improves the prediction of the measured HRTF only if LSD(R5) - LSD(R1) <= -1% of "
             "R0's LSD (about -0.046 dB), its CI is below 0 and p_C1 <= 0.05; otherwise adding REST does not improve the "
             'full-band prediction by 1% or more (this alone does not say that REST carries no low-frequency information)'),
    secondary=[
        'dR2_low = CV-R2(R5) - CV-R2(R1), measured latent, 0.2-5 kHz columns, SSE and SST summed over subjects; subject '
        'bootstrap CI and C1 null; read as low-band information only if >= +0.02, CI above 0 and above the C1 95th percentile',
        'ratio dR2_low,meas / dR2_low,sim (same bootstrap draws), predicted about 1/3.1 = 0.32; interpreted only if the CI of '
        'dR2_low,sim is above 0: a CI containing 0.32 supports measured = shape part + independent rest, one clearly above contradicts it',
        'descriptive: per-band R2 and LSD of every row on both targets, w per fold, REST point counts, HEADM distributions'],
    bands=('Task 1: 0.2-5 kHz bins 2-26, 6-12 kHz bins 32-64, 12-16 kHz bins 64-85 (limits included); '
           '5-6 kHz bins 27-31 descriptive; full-band statistics use all bins'),
    expectation='full-band dLSD within +-0.01 dB; measured dR2_low below +0.02; the simulated target may gain more',
    data='development only; the 36 locked subjects were spent on 2026-10-12 (FFM_3_8) and are not read',
)
print(json.dumps(REGISTRATION, indent=1))


In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（GELU + FiLM）
#
# 与 FFM48k_Train.ipynb 的模型定义逐字相同（构建时复制进来）。
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — 载入 FFM48k_measim
# ==========================================================================
ck = torch.load(PATHS['ckpt'], map_location='cpu', weights_only=False)
cfg = ck['config']
assert ck['run_name'] == 'FFM48k_measim' and cfg['num_bins'] == N_BINS and cfg['num_datasets'] == 6
assert cfg['data_zip_sha256'] == ZIP_SHA256 and cfg['split_sha256'] == SPLIT_SHA256
model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32, num_datasets=6,
                             num_bins=N_BINS, tie_encoder_film=True).to(device)
model.load_state_dict(ck['model_state_dict']); model.eval()
FILM_ID = {d['tag']: d['id'] for d in ck['datasets']}
assert FILM_ID['SONICOM'] == DM and FILM_ID['SONICOM_SIM'] == DS
print(f"FFM48k_measim epoch {ck['epoch'] + 1} | val {ck['val_rmse']:.4f} dB | FiLM 行 {FILM_ID}")
del ck

In [ ]:
# ==========================================================================
# Cell 4 — 回归引擎：核 ridge 的对偶形式（Task 1 Cell 4），kernel 由函数按训练行生成
#
# Kfun(ref) 返回全部行的 kernel，但标准化、常数列筛选和 kernel 定标只用 ref 行（训练折）估计。
# 复现关卡用 Kfun = lambda ref: K_global（Task 1 的做法：在全部开发行上估计）。
# ==========================================================================
def subject_folds(subjects, k, rng):
    p = rng.permutation(subjects)
    return [np.sort(a) for a in np.array_split(p, min(k, len(p)))]


def make_plan(ns, seed=SEED):
    # FFM_3_2_1 的 build_plan：随机数的调用顺序相同，所以折相同
    rng = np.random.default_rng(seed)
    subj = np.repeat(np.arange(ns), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    plan = []
    for te in subject_folds(np.arange(ns), N_FOLDS, rng):
        tr = np.setdiff1d(np.arange(ns), te)
        inner = [(rows(np.setdiff1d(tr, v)), rows(v)) for v in subject_folds(tr, N_INNER, rng)]
        plan.append(((rows(tr), rows(te)), inner))
    return plan


def centred_pair(G, tr, va):
    Gtt, Gvt = G[np.ix_(tr, tr)], G[np.ix_(va, tr)]
    cm, tot = Gtt.mean(0), Gtt.mean()
    return Gtt - cm[None] - cm[:, None] + tot, Gvt - cm[None] - Gvt.mean(1)[:, None] + tot


def centred_vv_diag(G, tr, va):
    r = G[np.ix_(va, tr)].mean(1)
    return np.diag(G[np.ix_(va, va)]) - 2 * r + G[np.ix_(tr, tr)].mean()


def weights(K, tr, va, alphas):
    Ktt, Kvt = centred_pair(K, tr, va)
    lam, Q = np.linalg.eigh(Ktt)
    lam = np.clip(lam, 0, None)
    KvQ, scale = Kvt @ Q, np.trace(Ktt) / len(tr)
    return [(KvQ / (lam + a * scale)) @ Q.T for a in alphas]


def row_sse(W, G, tr, va):
    CYtt, CYvt = centred_pair(G, tr, va)
    return centred_vv_diag(G, tr, va) - 2 * np.einsum('ij,ij->i', W, CYvt) + np.einsum('ij,ij->i', W @ CYtt, W)


def run_cv(Kfun, Y, plan, keep_pred=True):
    G = Y @ Y.T
    n = len(Y)
    sse, sst = np.zeros(n), np.zeros(n)
    pred = np.zeros_like(Y) if keep_pred else None
    chosen = []
    for (tr, va), inner in plan:
        tot = np.zeros(len(ALPHA_REL))
        for itr, iva in inner:
            for j, W in enumerate(weights(Kfun(itr), itr, iva, ALPHA_REL)):
                tot[j] += row_sse(W, G, itr, iva).sum()
        j = int(np.argmin(tot)); chosen.append(j)
        W = weights(Kfun(tr), tr, va, [ALPHA_REL[j]])[0]
        sse[va] = row_sse(W, G, tr, va); sst[va] = centred_vv_diag(G, tr, va)
        if keep_pred:
            my = Y[tr].mean(0)
            pred[va] = my + W @ (Y[tr] - my)
    return dict(r2=float(1 - sse.sum() / sst.sum()), sse=sse, sst=sst, alpha_idx=chosen, pred=pred)


def block_gram(X, ref):
    # 一个特征块：只用 ref 行估计均值、标准差和常数列；除以列数（多块拼接时每块权重相同）
    mu, sd = X[ref].mean(0), X[ref].std(0)
    keep = sd > 1e-12 * max(sd.max(), 1e-300)
    Xz = (X[:, keep] - mu[keep]) / sd[keep]
    return Xz @ Xz.T / Xz.shape[1]


def unit_scaled(K, ref):
    # kernel 定标：ref 行上中心化后的平均对角元 = 1（Task 1 的 unit_trace 只在 ref 上算）
    n = len(ref); C = np.eye(n) - 1.0 / n
    return K / (np.trace(C @ K[np.ix_(ref, ref)] @ C) / n)


def source_kernel(blocks, ref):
    return unit_scaled(sum(block_gram(X, ref) for X in blocks.values()), ref)


def kernel_fn(sources, fold_internal):
    if fold_internal:
        return lambda ref: sum(source_kernel(S, ref) for S in sources)
    allrows = np.arange(len(next(iter(sources[0].values()))))
    K = sum(source_kernel(S, allrows) for S in sources)
    return lambda ref: K


def fit_predict(Kfun, Y_dev, n_dev, n_all, seed=SEED):
    # 测试用：α 由开发集上的 5 折按人分组 CV 选，再用全部开发行拟合，预测其余行
    G = Y_dev @ Y_dev.T
    rng = np.random.default_rng(seed); subj = np.repeat(np.arange(n_dev // 2), 2)
    rows = lambda s: np.flatnonzero(np.isin(subj, s))
    tot = np.zeros(len(ALPHA_REL))
    for v in subject_folds(np.arange(n_dev // 2), N_INNER, rng):
        itr, iva = rows(np.setdiff1d(np.arange(n_dev // 2), v)), rows(v)
        for j, W in enumerate(weights(Kfun(itr), itr, iva, ALPHA_REL)):
            tot[j] += row_sse(W, G, itr, iva).sum()
    j = int(np.argmin(tot))
    dev, te = np.arange(n_dev), np.arange(n_dev, n_all)
    W = weights(Kfun(dev), dev, te, [ALPHA_REL[j]])[0]
    my = Y_dev.mean(0)
    return my + W @ (Y_dev - my), j


# 自检：随机特征的 CV-R² 应在 0 附近或为负；两种标准化都要过
_plan = make_plan(40)
_Y = np.random.default_rng(3).standard_normal((80, 50)); _S = [{'x': np.random.default_rng(4).standard_normal((80, 30))}]
for fi in (False, True):
    print(f"自检（fold_internal={fi}）：随机特征 CV-R² = {run_cv(kernel_fn(_S, fi), _Y, _plan, keep_pred=False)['r2']:+.4f}（应 ≤ ~0）")

In [ ]:
# ==========================================================================
# Cell 5 — 数据 · latent · 几何特征 · 指标
# ==========================================================================
def load_rows(tag, ids):
    # 每人两行（左耳, 镜像后的右耳）；每个数组按 manifest 核对 SHA-256
    d = f'{DATA}/{tag}_48k'
    man = {m['id']: m for m in json.load(open(f'{d}/manifest.json'))}
    X = np.empty((2 * len(ids), N_BINS, 3072), np.float32)
    for k, s in enumerate(ids):
        for side in ('left', 'right'):
            f = f'{d}/npy_{side}/{man[s][side]}'
            assert sha256(f) == man[s][f'{side}_sha256'], f
            a = np.load(f)[:N_BINS]
            X[2 * k + (side == 'right')] = a[:, MIRROR] if side == 'right' else a
    return X


def encode(X, did):
    Z = []
    for b in range(0, len(X), BS):
        x = torch.from_numpy(X[b:b + BS]).to(device)
        _, z = model(x, torch.full((len(x),), did, dtype=torch.long, device=device))
        Z.append(z.reshape(len(x), -1).double().cpu().numpy())
    return np.concatenate(Z)


def geometry_blocks(sids):
    # FFM_3_2_1 Cell 4/5（= Task 1 sonicom_geometry），只返回原始特征块，不做标准化
    man = {f"SON_{r['subject']}": r for r in csv.DictReader(open(f'{SAMPLES}/manifest.csv'))}
    flip = np.array([1.0, -1.0, 1.0])
    G0T = torch.as_tensor(np.random.default_rng(12345).standard_normal((M_RFF, 3)).T, dtype=torch.float32, device=device)

    def rff(P):
        base = 2 * np.pi * (torch.as_tensor(P, dtype=torch.float32, device=device) @ G0T)
        return {sg: torch.cat([torch.cos(sg * base).mean(0), torch.sin(sg * base).mean(0)]).double().cpu().numpy()
                for sg in SIGMAS}

    pct = lambda a, q: float(np.percentile(a, q))
    simple, ear = [], []
    for s in sids:
        d = np.load(f'{SAMPLES}/{s[4:]}.npz')
        head = d['head_pts'][:20000].astype(np.float64)
        head = head[head[:, 2] > -75.0]
        for side in (0, 1):
            if side == 0:
                pts, canal, area, hd = d['earL_pts'], d['ear_left_mm'].astype(np.float64), float(man[s]['crop_area_L_mm2']), head
            else:
                pts, canal, area, hd = d['earR_pts'], d['ear_right_mm'].astype(np.float64) * flip, float(man[s]['crop_area_R_mm2']), head * flip
            assert canal[1] > 0
            q = pts[:20000].astype(np.float64) - canal
            out = q[q[:, 1] > 3.0]
            simple.append([canal[0], canal[1], canal[2], pct(q[:, 1], 95),
                           pct(out[:, 2], 95) - pct(out[:, 2], 5), pct(out[:, 0], 95) - pct(out[:, 0], 5),
                           area, pct(hd[:, 0], 99.5) - pct(hd[:, 0], 0.5), pct(hd[:, 2], 99.9)])
            ear.append(rff(q))
    blocks = {'simple': np.array(simple)}
    for sg in SIGMAS:
        blocks[f'ear_s{sg}'] = np.stack([f[sg] for f in ear])
    return blocks


M45 = torch.as_tensor(MEAS_DIRS, device=device)


def metrics(pred, X):
    """pred(b0, b1) -> (b, 86, 3072) 张量；X 是实测（numpy）。返回逐耳的各项指标"""
    out = {k: [] for k in ('se_all', 'lsd_el45', 'lsd_full', 'mae_el45', 'mae_full', 'sd_bin_el45', 'sd_bin_full')}
    for b0 in range(0, len(X), BS):
        b1 = min(b0 + BS, len(X))
        e = (pred(b0, b1).float() - torch.from_numpy(X[b0:b1]).to(device)).double()
        out['se_all'].append((e ** 2).sum((1, 2)))                    # 旧口径：全部 bin、全部方向
        eb = e[:, EVAL_BINS]
        d = torch.sqrt((eb ** 2).mean(1))                             # 每个方向沿频率 RMS
        out['lsd_full'].append(d.mean(1)); out['lsd_el45'].append(d[:, M45].mean(1))
        a = eb.abs()
        out['mae_full'].append(a.mean((1, 2))); out['mae_el45'].append(a[:, :, M45].mean((1, 2)))
        out['sd_bin_el45'].append(torch.sqrt((eb[:, :, M45] ** 2).mean(2)))
        out['sd_bin_full'].append(torch.sqrt((eb ** 2).mean(2)))
    return {k: torch.cat(v).cpu().numpy() for k, v in out.items()}


def per_subject(m):
    ns = len(m['se_all']) // 2
    S = {k: m[k].reshape(ns, 2).mean(1) for k in ('lsd_el45', 'lsd_full', 'mae_el45', 'mae_full')}
    S['rmse_all'] = np.sqrt(m['se_all'].reshape(ns, 2).sum(1) / (2 * N_BINS * 3072))
    return S


def summary(m):
    S = per_subject(m)
    r = {k: float(v.mean()) for k, v in S.items() if k != 'rmse_all'}
    r.update({f'sd_{k}': float(S[k].std(ddof=1)) for k in ('lsd_el45', 'mae_el45')})
    r['pooled_rmse'] = float(np.sqrt(m['se_all'].sum() / (len(m['se_all']) * N_BINS * 3072)))
    r['curve_el45'] = m['sd_bin_el45'].mean(0).tolist(); r['curve_full'] = m['sd_bin_full'].mean(0).tolist()
    return r


def decoder_pred(Zrows, did):
    return lambda b0, b1: model.decoder(torch.from_numpy(Zrows[b0:b1].reshape(-1, N_BINS, 64)).float().to(device),
                                        torch.full((b1 - b0,), did, dtype=torch.long, device=device))


def ae_pred(X, did):
    return lambda b0, b1: model(torch.from_numpy(X[b0:b1]).to(device),
                                torch.full((b1 - b0,), did, dtype=torch.long, device=device))[0]


def paired(a, b, rng):
    """a − b over subjects（负 = a 更好）：均值、bootstrap 95% CI、单侧配对 t 检验 p（H1: a < b）"""
    d = a - b
    bs = np.array([d[rng.integers(0, len(d), len(d))].mean() for _ in range(N_BOOT)])
    return dict(diff=float(d.mean()), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)),
                t_p_one_sided=float(stats.ttest_rel(a, b, alternative='less').pvalue))


MASUYAMA = {'Avg. HRTF': dict(mae=3.78, lsd=5.05), 'Mesh2HRTF (raw simulation)': dict(mae=7.50, lsd=10.53),
            'S2RNF': dict(mae=3.60, lsd=4.90)}     # 他们的表 1（SONICOM 25 人测试集；"RMSE" 即 LSD，到 20 kHz、不含 DC、793 个测点）


def show(title, rows, mean_key='training mean'):
    print(f'\n{title}')
    print(f"{'':<44}{'R²(z)':>8}{'LSD≥−45°':>10}{'vs均值':>8}{'MAE≥−45°':>10}{'LSD全球':>9}{'vs均值':>8}{'旧pooled':>10}")
    m0 = rows[mean_key]
    for k, r in rows.items():
        rel = lambda key: f"{(m0[key] - r[key]) / m0[key]:+8.1%}"
        r2 = f"{r['r2']:+8.4f}" if r.get('r2') is not None else f"{'':>8}"
        print(f"{k:<44}{r2}{r['lsd_el45']:10.3f}{rel('lsd_el45')}{r['mae_el45']:10.3f}{r['lsd_full']:9.3f}{rel('lsd_full')}{r['pooled_rmse']:10.3f}")
    print('  参照（Masuyama 2026 表 1，口径不同，只比"相对均值"）：' +
          '；'.join(f"{k} LSD {v['lsd']:.2f} / MAE {v['mae']:.2f}" for k, v in MASUYAMA.items()) +
          f"；S2RNF 相对均值 LSD {(5.05 - 4.90) / 5.05:+.1%}，MAE {(3.78 - 3.60) / 3.78:+.1%}")

In [ ]:
# ==========================================================================
# Cell 6 — 开发集：受试者、折、数据、latent、几何
# ==========================================================================
L = np.load(PATHS['subject_lists'])          # 只用 eval_sids / locked_sids
DEV = [str(s) for s in L['eval_sids']]; LOCKED = [str(s) for s in L['locked_sids']]
assert len(DEV) == 164 and len(LOCKED) == 36 and not set(DEV) & set(LOCKED)
assert set(DEV) <= set(SPLIT['SONICOM']['train']) | set(SPLIT['SONICOM']['val'])
assert set(LOCKED) <= set(SPLIT['SONICOM']['test'])
O = np.load(PATHS['fold_ref'], allow_pickle=True)   # 只用 dev / fold_of_subject
assert [str(s) for s in O['dev']] == DEV
PLAN = make_plan(len(DEV))
fold = np.full(len(DEV), -1)
for f, ((tr, va), _) in enumerate(PLAN):
    fold[np.unique(va // 2)] = f
assert np.array_equal(fold, O['fold_of_subject']), '折与 FFM_3_2_1 不同'
FOLD_ROW = np.repeat(fold, 2)
print(f'开发集 {len(DEV)} 人 / {2 * len(DEV)} 只耳，外层折与 FFM_3_2_1 相同；锁定 {len(LOCKED)} 人（本格不读）')

t0 = time.time()
X_meas, X_sim = load_rows('SONICOM', DEV), load_rows('SONICOM_SIM', DEV)
Zm, Zs = encode(X_meas, DM), encode(X_sim, DS)
GEO = geometry_blocks(DEV)
print(f'数据、latent、几何 {time.time() - t0:.0f}s | z {Zm.shape} | 几何块 ' + ', '.join(f'{k} {v.shape[1]}' for k, v in GEO.items()))
SOURCES = {'geometry': GEO, 'simulation': {'z_sim': Zs}}

# 训练折均值（实测、仿真）：每折一个
MU_M = np.stack([X_meas[tr].astype(np.float64).mean(0) for (tr, _), _ in PLAN]).astype(np.float32)
MU_S = np.stack([X_sim[tr].astype(np.float64).mean(0) for (tr, _), _ in PLAN]).astype(np.float32)
fold_mean = lambda MU: (lambda b0, b1: torch.from_numpy(MU[FOLD_ROW[b0:b1]]).to(device))
REF_PRED = {
    'training mean': fold_mean(MU_M),
    'raw simulation': lambda b0, b1: torch.from_numpy(X_sim[b0:b1]).to(device),
    'simulated deviation + measured mean': lambda b0, b1: torch.from_numpy(X_sim[b0:b1] - MU_S[FOLD_ROW[b0:b1]] + MU_M[FOLD_ROW[b0:b1]]).to(device),
    'AE reconstruction (reference)': ae_pred(X_meas, DM),
}
REF = {k: metrics(p, X_meas) for k, p in REF_PRED.items()}

In [ ]:
# ==========================================================================
# Cell 7 — 复现关卡：R0 和 R1（GEO）必须逐位复现 FFM_3_8 的开发集结果（每折内部标准化）
# 期望值：mesh2z_48k_v1/dev_results.json，FFM_3_9 的关卡也复现过（训练集均值 4.558416，几何 4.377719，CV-R² 0.069439）
# ==========================================================================
t0 = time.time()
CV_R1 = run_cv(kernel_fn([GEO], fold_internal=True), Zm, PLAN)
S_R1 = summary(metrics(decoder_pred(CV_R1['pred'], DM), X_meas))
S_R0 = summary(REF['training mean'])
GATE = dict(r0_lsd=S_R0['lsd_el45'], r1_lsd=S_R1['lsd_el45'], r1_r2=CV_R1['r2'])
EXPECT = dict(r0_lsd=4.558416, r1_lsd=4.377719, r1_r2=0.069439)
print(f'关卡（{time.time() - t0:.0f}s）：' + '，'.join(f'{k} {GATE[k]:.6f}（期望 {EXPECT[k]:.6f}）' for k in EXPECT))
bad = {k: (GATE[k], EXPECT[k]) for k in EXPECT if abs(GATE[k] - EXPECT[k]) > 5e-5}
assert not bad, f'复现不了 FFM_3_8 的开发集结果 {bad} —— 数据、折、编码器或引擎不对，停止'
THRESH = 0.01 * S_R0['lsd_el45']
print(f'✓ 关卡通过。判读门槛 = 训练集均值 LSD 的 1% = {THRESH:.4f} dB')

In [ ]:
# ==========================================================================
# Cell 8 — REST 和 HEADM 特征（登记：PROJECT_LOG §2C 2026-10-23）
# 输出格式与 GEO 相同：dict，块名 -> (2·len(sids), d)，行顺序 = 每人 [左耳, 镜像后的右耳]
# ==========================================================================
from scipy.spatial import ConvexHull

FLIP = np.array([1.0, -1.0, 1.0])
HEADM_NAMES = ['width at ear level (incl. pinnae)', 'back of head', 'front of head', 'hull volume (cm3)', 'canal x - median head x']


def rff_mean(P):
    # 与 geometry_blocks 里的 rff() 相同：同一组随机方向（种子 12345）、同样的 5 个尺度，cos / sin 对点取平均
    base = 2 * np.pi * (torch.as_tensor(P, dtype=torch.float32, device=device) @ _G0T)
    return {sg: torch.cat([torch.cos(sg * base).mean(0), torch.sin(sg * base).mean(0)]).double().cpu().numpy()
            for sg in SIGMAS}


def rest_points(d):
    """全部 head_pts 里 z > Z_CUT、且离两个耳道都超过 EAR_R 的点，取前 N_REST 个（点是随机顺序，前 k 个即均匀子样本）"""
    head = d['head_pts'].astype(np.float64)
    cL, cR = d['ear_left_mm'].astype(np.float64), d['ear_right_mm'].astype(np.float64)
    keep = (head[:, 2] > Z_CUT) & (np.linalg.norm(head - cL, axis=1) > EAR_R) & (np.linalg.norm(head - cR, axis=1) > EAR_R)
    P = head[keep][:N_REST]
    assert np.linalg.norm(P - cL, axis=1).min() > EAR_R and np.linalg.norm(P - cR, axis=1).min() > EAR_R
    assert P[:, 2].min() > Z_CUT
    return P, int(keep.sum())


def rest_blocks(sids):
    feats, info = [], []
    for s in sids:
        d = np.load(f'{SAMPLES}/{s[4:]}.npz')
        P, n_kept = rest_points(d)
        assert d['ear_left_mm'][1] > 0 and (d['ear_right_mm'] * FLIP)[1] > 0    # 镜像后，原右耳道在 +y（左耳一侧）
        feats += [rff_mean(P), rff_mean(P * FLIP)]                                # 左耳行：原样；右耳行：整个头 y -> -y
        info.append(dict(sid=s, n_head=int(len(d['head_pts'])), n_kept=n_kept, n_used=int(len(P)),
                         z_top=float(np.percentile(P[:, 2], 99.9)), z_median=float(np.median(P[:, 2]))))
    return {f'rest_s{sg}': np.stack([f[sg] for f in feats]) for sg in SIGMAS}, info


def head_measures(sids):
    pct = lambda a, q: float(np.percentile(a, q))
    rows = []
    for s in sids:
        d = np.load(f'{SAMPLES}/{s[4:]}.npz')
        head = d['head_pts'].astype(np.float64)
        head = head[head[:, 2] > Z_CUT]
        mid = head[np.abs(head[:, 2]) < 20.0]
        common = [pct(mid[:, 1], 99.5) - pct(mid[:, 1], 0.5), -pct(head[:, 0], 0.5), pct(head[:, 0], 99.5),
                  ConvexHull(head).volume / 1000.0]
        mx = float(np.median(head[:, 0]))
        for canal in (d['ear_left_mm'], d['ear_right_mm']):    # x 不受镜像影响；每行用自己那只耳的耳道
            rows.append(common + [float(canal[0]) - mx])
    return {'headm': np.array(rows)}


def robust_z(a):
    a = np.asarray(a, float)
    return (a - np.median(a)) / (1.4826 * np.median(np.abs(a - np.median(a))) + 1e-12)


# ---- 运行
_G0T = torch.as_tensor(np.random.default_rng(12345).standard_normal((M_RFF, 3)).T, dtype=torch.float32, device=device)
t0 = time.time()
REST, REST_INFO = rest_blocks(DEV)
HEADM = head_measures(DEV)
SRC = {'GEO': GEO, 'REST': REST, 'HEADM': HEADM}
assert all(len(X) == 2 * len(DEV) for S in SRC.values() for X in S.values())
print(f'REST 和 HEADM（{time.time() - t0:.0f}s）| REST 块 ' + ', '.join(f'{k} {v.shape[1]}' for k, v in REST.items()))

n_kept = np.array([r['n_kept'] for r in REST_INFO]); n_used = np.array([r['n_used'] for r in REST_INFO])
z_top = np.array([r['z_top'] for r in REST_INFO])
OUTLIERS = dict(z_top=[REST_INFO[i]['sid'] for i in np.flatnonzero(np.abs(robust_z(z_top)) > 3)],
                n_kept=[REST_INFO[i]['sid'] for i in np.flatnonzero(np.abs(robust_z(n_kept)) > 3)])
print(f'REST 点数：过滤后中位 {np.median(n_kept):.0f}（最少 {n_kept.min()}），每人用 {N_REST} 个；不足 {N_REST} 的 {int((n_used < N_REST).sum())} 人')
print(f'REST 头顶高度（z 的 99.9 百分位）中位 {np.median(z_top):.1f} mm，IQR [{np.percentile(z_top, 25):.1f}, {np.percentile(z_top, 75):.1f}]；'
      f'离群（稳健 z > 3）：头顶 {OUTLIERS["z_top"] or "无"}；点数 {OUTLIERS["n_kept"] or "无"}')
H = HEADM['headm']
HEADM_STATS = {n: dict(median=float(np.median(H[::2, i])), q25=float(np.percentile(H[::2, i], 25)),
                       q75=float(np.percentile(H[::2, i], 75))) for i, n in enumerate(HEADM_NAMES)}
print('HEADM（每人一行，左耳）：')
for n, v in HEADM_STATS.items():
    print(f'  {n:<33} 中位 {v["median"]:8.1f}，IQR [{v["q25"]:.1f}, {v["q75"]:.1f}]')


In [ ]:
# ==========================================================================
# Cell 9 — 各行交叉验证，实测和仿真两个目标（登记的表：R1–R7；R5 为主行）
# ==========================================================================
def cached(Kfun):
    # 同一组训练行的 kernel 只算一次（各行、两个目标和 C1 共用）
    memo = {}

    def f(ref):
        key = ref.tobytes()
        if key not in memo:
            memo[key] = Kfun(ref)
        return memo[key]
    return f


def run_cv_w(Ka, Kb, Y, plan, w_grid=W_GRID, keep_pred=True):
    """R5：kernel = Ka(ref) + w·Kb(ref)。每个外层训练折内部，用同一组内层折同时选 (w, α)，使 latent SSE 最小。
    SSE 的算法与 run_cv / row_sse 相同，只是每个内层折的中心化 G 只算一次"""
    G = Y @ Y.T
    n = len(Y)
    sse, sst = np.zeros(n), np.zeros(n)
    pred = np.zeros_like(Y) if keep_pred else None
    chosen = []
    for (tr, va), inner in plan:
        tot = np.zeros((len(w_grid), len(ALPHA_REL)))
        for itr, iva in inner:
            A, B = Ka(itr), Kb(itr)
            CYtt, CYvt = centred_pair(G, itr, iva)
            s0 = centred_vv_diag(G, itr, iva)
            for i, w in enumerate(w_grid):
                for j, W in enumerate(weights(A + w * B, itr, iva, ALPHA_REL)):
                    tot[i, j] += (s0 - 2 * np.einsum('ij,ij->i', W, CYvt) + np.einsum('ij,ij->i', W @ CYtt, W)).sum()
        i, j = np.unravel_index(int(np.argmin(tot)), tot.shape)
        chosen.append(dict(w=float(w_grid[i]), alpha_idx=int(j)))
        W = weights(Ka(tr) + w_grid[i] * Kb(tr), tr, va, [ALPHA_REL[j]])[0]
        sse[va] = row_sse(W, G, tr, va); sst[va] = centred_vv_diag(G, tr, va)
        if keep_pred:
            my = Y[tr].mean(0)
            pred[va] = my + W @ (Y[tr] - my)
    return dict(r2=float(1 - sse.sum() / sst.sum()), sse=sse, sst=sst, chosen=chosen, pred=pred)


ZCOLS = {'all': np.ones(N_BINS * 64, bool), **{b: np.repeat(m, 64) for b, m in BIN_BAND.items()}}   # z 的列 = bin·64 + 维


def band_parts(cv, Y, plan):
    """逐行、逐频带的 (SSE, SST)；SST 相对该行所在外层折的训练均值 —— 与 run_cv 的 sse / sst、Task 1 的分频带 R² 同一口径"""
    E, C = (Y - cv['pred']) ** 2, np.zeros_like(Y)
    for (tr, va), _ in plan:
        C[va] = (Y[va] - Y[tr].mean(0)) ** 2
    parts = {b: (E[:, c].sum(1), C[:, c].sum(1)) for b, c in ZCOLS.items()}
    assert np.allclose(parts['all'][0], cv['sse'], rtol=1e-6) and np.allclose(parts['all'][1], cv['sst'], rtol=1e-9)
    return parts


def r2_of(p):
    return float(1 - p[0].sum() / p[1].sum())


def subj(a):
    return a.reshape(-1, 2).mean(1)            # 每人 = 两只耳的平均


def lsd_parts(pred, X):
    """逐耳 LSD，仰角 ≥ −45°：'all' = bin 1–85（与 metrics() 的 lsd_el45 相同），以及各频带的 bin"""
    out = {b: [] for b in BAND_BINS}
    for b0 in range(0, len(X), BS):
        b1 = min(b0 + BS, len(X))
        e = (pred(b0, b1).float() - torch.from_numpy(X[b0:b1]).to(device)).double()[:, :, M45]
        for b, k in BAND_BINS.items():
            out[b].append(torch.sqrt((e[:, k] ** 2).mean(1)).mean(1))
    return {b: torch.cat(v).cpu().numpy() for b, v in out.items()}


# ---- 运行
BAND_BINS = {'all': torch.arange(1, N_BINS, device=device),
             **{b: torch.as_tensor(np.flatnonzero(m), device=device) for b, m in BIN_BAND.items()}}
COLS = ['all', '0.2-5 kHz', '5-6 kHz (no band)', '6-12 kHz', '12-16 kHz']
KS = {name: cached(lambda ref, S=S: source_kernel(S, ref)) for name, S in SRC.items()}


def kernel_of(srcs):
    return lambda ref: sum(KS[s](ref) for s in srcs)       # = kernel_fn([...], fold_internal=True)


# 关卡 2：缓存的 GEO kernel 复现 Cell 7 的 R1；R5 的代码在 w 网格只有 0 时退化为 R1
_c = run_cv(kernel_of(['GEO']), Zm, PLAN, keep_pred=False)
assert np.allclose(_c['sse'], CV_R1['sse'], rtol=1e-12) and _c['alpha_idx'] == CV_R1['alpha_idx']
_w0 = run_cv_w(KS['GEO'], KS['REST'], Zm, PLAN, w_grid=[0.0], keep_pred=False)
assert np.allclose(_w0['sse'], CV_R1['sse'], rtol=1e-9) and [c['alpha_idx'] for c in _w0['chosen']] == CV_R1['alpha_idx']
print('✓ 关卡 2：R1 复现；w = 0 时 R5 的选择与 R1 相同')

TARGETS = {'meas': dict(Y=Zm, X=X_meas, did=DM, MU=MU_M), 'sim': dict(Y=Zs, X=X_sim, did=DS, MU=MU_S)}
FIXED = {'R1 GEO': ['GEO'], 'R2 GEO + REST': ['GEO', 'REST'], 'R3 GEO + HEADM': ['GEO', 'HEADM'],
         'R4 GEO + REST + HEADM': ['GEO', 'REST', 'HEADM'], 'R6 REST only': ['REST'], 'R7 HEADM only': ['HEADM']}
PRIMARY = 'R5 GEO + w·REST'
R0 = 'R0 training mean'
ORDER = [R0, 'R1 GEO', PRIMARY, 'R2 GEO + REST', 'R3 GEO + HEADM', 'R4 GEO + REST + HEADM', 'R6 REST only', 'R7 HEADM only']

RES, PARTS, LSD, SUMMARY = {}, {}, {}, {}
t0 = time.time()
for t, T in TARGETS.items():
    RES[t] = {name: run_cv(kernel_of(srcs), T['Y'], PLAN) for name, srcs in FIXED.items()}
    RES[t][PRIMARY] = run_cv_w(KS['GEO'], KS['REST'], T['Y'], PLAN)
    PARTS[t] = {name: band_parts(cv, T['Y'], PLAN) for name, cv in RES[t].items()}
    LSD[t] = {R0: lsd_parts(fold_mean(T['MU']), T['X'])}
    LSD[t].update({name: lsd_parts(decoder_pred(cv['pred'], T['did']), T['X']) for name, cv in RES[t].items()})
    print(f'目标 {t}：{time.time() - t0:.0f}s')
SUMMARY = {name: summary(metrics(decoder_pred(RES['meas'][name]['pred'], DM), X_meas)) for name in ORDER[1:]}
SUMMARY[R0] = S_R0

# 一致性：lsd_parts 的全频段 = metrics() 的 lsd_el45；R0、R1 = 关卡；R1 的低频 R² ≈ Task 1（0.1234 实测、0.4955 仿真）
assert abs(subj(LSD['meas']['R1 GEO']['all']).mean() - S_R1['lsd_el45']) < 1e-6
assert abs(subj(LSD['meas'][R0]['all']).mean() - S_R0['lsd_el45']) < 1e-6
assert abs(SUMMARY['R1 GEO']['lsd_el45'] - S_R1['lsd_el45']) < 1e-6
r1_low = {t: r2_of(PARTS[t]['R1 GEO'][LOW]) for t in TARGETS}
assert abs(r1_low['meas'] - 0.12340) < 0.005, f"R1 的 0.2–5 kHz R² {r1_low['meas']:.4f} 与 Task 1 的 0.1234 不符：查分频带代码"
if abs(r1_low['sim'] - 0.49552) > 0.005:
    print(f"★ 注意：仿真目标 R1 的 0.2–5 kHz R² {r1_low['sim']:.4f}，Task 1 是 0.4955（Task 1 在全部开发行上标准化）")
print(f"R1 的 0.2–5 kHz CV-R²：实测 {r1_low['meas']:.4f}（Task 1 0.1234），仿真 {r1_low['sim']:.4f}（Task 1 0.4955）")

for t in TARGETS:
    print(f'\n目标：{"实测" if t == "meas" else "仿真"} latent。CV-R²(z) 与 LSD ≥ −45°（dB），列：' + ' / '.join(COLS))
    for name in ORDER:
        r2s = '  '.join(f'{r2_of(PARTS[t][name][c]):+.3f}' for c in COLS) if name in PARTS[t] else ' ' * 38
        lsds = '  '.join(f'{subj(LSD[t][name][c]).mean():.3f}' for c in COLS)
        print(f'  {name:<24} R² {r2s}  | LSD {lsds}')
    print(f"  R5 每折选到的 w：{[c['w'] for c in RES[t][PRIMARY]['chosen']]}")


In [ ]:
# ==========================================================================
# Cell 10 — 主检验 · C1 置换（100 次）· 次要统计量 · 判读（规则见 REGISTRATION，事先定好）
# ==========================================================================
def perm_rows(p):
    # 第 i 人的两行（左、右）<- 第 p[i] 人的左、右两行：两只耳一起动，左耳行仍对左耳行
    return (2 * np.asarray(p)[:, None] + np.arange(2)).ravel()


def boot_r2(p, B):
    # 按人 bootstrap 的 R²：每次重采样受试者，SSE 与 SST 先求和再相除（两只耳在同一人里）
    s, c = p[0].reshape(-1, 2).sum(1), p[1].reshape(-1, 2).sum(1)
    return 1 - s[B].sum(1) / c[B].sum(1)


def ci95(a):
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]


# ---- 运行
ns = len(DEV)
L1, L5 = subj(LSD['meas']['R1 GEO']['all']), subj(LSD['meas'][PRIMARY]['all'])
PRIM = paired(L5, L1, np.random.default_rng(SEED))
DR2 = {t: r2_of(PARTS[t][PRIMARY][LOW]) - r2_of(PARTS[t]['R1 GEO'][LOW]) for t in TARGETS}

# C1：REST 在受试者之间打乱，其余照 R5（每次重选 w 和 α）
NULL = dict(dlsd=[], dr2_low=[], w=[])
t0 = time.time()
for k in range(N_NULL):
    rows = perm_rows(np.random.default_rng(k).permutation(ns))
    REST_k = {b: X[rows] for b, X in REST.items()}
    cv = run_cv_w(KS['GEO'], lambda ref: source_kernel(REST_k, ref), Zm, PLAN)
    NULL['dlsd'].append(float(subj(lsd_parts(decoder_pred(cv['pred'], DM), X_meas)['all']).mean() - L1.mean()))
    NULL['dr2_low'].append(r2_of(band_parts(cv, Zm, PLAN)[LOW]) - r2_of(PARTS['meas']['R1 GEO'][LOW]))
    NULL['w'].append([c['w'] for c in cv['chosen']])
    if k in (0, 4, 19, 49) or k == N_NULL - 1:
        el = time.time() - t0
        print(f'  C1 {k + 1}/{N_NULL}：{el / 60:.1f} 分钟，预计还要 {el / (k + 1) * (N_NULL - k - 1) / 60:.0f} 分钟')
nd, nr = np.array(NULL['dlsd']), np.array(NULL['dr2_low'])
P_C1 = float((1 + (nd <= PRIM['diff']).sum()) / (N_NULL + 1))
P_C1_LOW = float((1 + (nr >= DR2['meas']).sum()) / (N_NULL + 1))

# 次要：低频 ΔR²（按人 bootstrap，实测和仿真用同一组重采样）与两者之比
B = np.random.default_rng(SEED).integers(0, ns, (N_BOOT, ns))
BD = {t: boot_r2(PARTS[t][PRIMARY][LOW], B) - boot_r2(PARTS[t]['R1 GEO'][LOW], B) for t in TARGETS}
LOWSTAT = {t: dict(diff=DR2[t], ci95=ci95(BD[t])) for t in TARGETS}
LOWSTAT['meas'].update(c1_p=P_C1_LOW, c1_95th=float(np.percentile(nr, 95)))
with np.errstate(divide='ignore', invalid='ignore'):
    RATIO = dict(value=float(DR2['meas'] / DR2['sim']) if DR2['sim'] != 0 else None,
                 ci95=ci95(BD['meas'] / BD['sim']), predicted=RATIO_PRED)
if LOWSTAT['sim']['ci95'][0] > 0:
    lo, hi = RATIO['ci95']
    RATIO['reading'] = ('CI contains 1/3.1: supports measured = shape part + independent rest' if lo <= RATIO_PRED <= hi else
                        'CI clearly above 1/3.1: contradicts the decomposition' if lo > RATIO_PRED else
                        'CI below 1/3.1 (not covered by the registered reading; descriptive)')
else:
    RATIO['reading'] = 'not interpreted: the CI of the simulated low-band gain includes 0'

# 次要（描述）：R2–R4 对 R1 的全频段 ΔLSD；R5 − R1 分频带的 ΔLSD
SECONDARY = {name: paired(subj(LSD['meas'][name]['all']), L1, np.random.default_rng(SEED))
             for name in ['R2 GEO + REST', 'R3 GEO + HEADM', 'R4 GEO + REST + HEADM']}
BAND_DLSD = {t: {c: float(subj(LSD[t][PRIMARY][c]).mean() - subj(LSD[t]['R1 GEO'][c]).mean()) for c in COLS} for t in TARGETS}

# 判读（事先定的规则）
PRIMARY_OK = PRIM['diff'] <= -THRESH and PRIM['hi'] < 0 and P_C1 <= 0.05
LOW_OK = DR2['meas'] >= 0.02 and LOWSTAT['meas']['ci95'][0] > 0 and DR2['meas'] > LOWSTAT['meas']['c1_95th']
VERDICT = dict(
    primary=('the head beyond the ear crops improves the prediction of the measured HRTF' if PRIMARY_OK else
             'adding REST does not improve the full-band prediction by 1% or more'),
    low_band=('REST adds low-band information on the measured latent' if LOW_OK else
              'no registered evidence that REST adds low-band information on the measured latent'),
    ratio=RATIO['reading'])

print(f"\n主检验 ΔLSD = LSD(R5) − LSD(R1) = {PRIM['diff']:+.4f} dB [{PRIM['lo']:+.4f}, {PRIM['hi']:+.4f}]，"
      f"单侧配对 t p = {PRIM['t_p_one_sided']:.3g}；C1 p = {P_C1:.3f}；门槛 −{THRESH:.4f} dB")
print(f"  C1 零分布（ΔLSD）：中位 {np.median(nd):+.4f}，5–95 百分位 [{np.percentile(nd, 5):+.4f}, {np.percentile(nd, 95):+.4f}]；"
      f"C1 里选到 w > 0 的折占 {np.mean(np.array(NULL['w']) > 0):.0%}")
print(f"  R5 实测每折的 w：{[c['w'] for c in RES['meas'][PRIMARY]['chosen']]}")
print(f"次要 ΔR²_low（实测）= {DR2['meas']:+.4f} {LOWSTAT['meas']['ci95']}，C1 95 百分位 {LOWSTAT['meas']['c1_95th']:+.4f}，C1 p = {P_C1_LOW:.3f}")
print(f"     ΔR²_low（仿真）= {DR2['sim']:+.4f} {LOWSTAT['sim']['ci95']}")
print(f"     比值 实测 / 仿真 = {RATIO['value']}，95% {RATIO['ci95']}（预测 {RATIO_PRED:.2f}）：{RATIO['reading']}")
for name, r in SECONDARY.items():
    print(f"描述 {name} − R1：{r['diff']:+.4f} dB [{r['lo']:+.4f}, {r['hi']:+.4f}]")
for t in TARGETS:
    print(f"描述 R5 − R1 分频带 ΔLSD（{t}）：" + '，'.join(f'{c} {v:+.4f}' for c, v in BAND_DLSD[t].items()))
print(f"\n判读（事先定的规则）：{VERDICT['primary']}。低频：{VERDICT['low_band']}。比值：{VERDICT['ratio']}")


In [ ]:
# ==========================================================================
# Cell 11 — 保存：results.json、per_subject_lsd.npz、per_band.png
# ==========================================================================
import matplotlib.pyplot as plt

ROWS_JSON = {t: {name: dict(
    cv_r2={c: r2_of(PARTS[t][name][c]) for c in COLS} if name in PARTS[t] else None,
    lsd_el45={c: float(subj(LSD[t][name][c]).mean()) for c in COLS},
    folds=(RES[t][name].get('chosen') or [dict(alpha_idx=int(j)) for j in RES[t][name]['alpha_idx']]) if name in RES[t] else None)
    for name in ORDER} for t in TARGETS}
RESULTS = dict(
    created=time.strftime('%Y-%m-%d %H:%M:%S'), notebook='mesh2z/FFM_3_10_Mesh2Z_HeadRest.ipynb',
    registration=REGISTRATION, gate=dict(GATE, expected=EXPECT), threshold_db=THRESH, n_subjects=len(DEV),
    bands={b: np.flatnonzero(m).tolist() for b, m in BIN_BAND.items()},
    features=dict(rest=REST_INFO, rest_outliers=OUTLIERS, headm_names=HEADM_NAMES, headm=HEADM_STATS),
    rows=ROWS_JSON, summary_meas=SUMMARY,
    primary=dict(PRIM, statistic='LSD(R5) - LSD(R1), dB', p_c1=P_C1, threshold=-THRESH),
    c1=dict(n=N_NULL, dlsd=NULL['dlsd'], dr2_low=NULL['dr2_low'], w=NULL['w']),
    low_band=LOWSTAT, ratio=RATIO, secondary=SECONDARY, band_dlsd=BAND_DLSD, verdict=VERDICT)
json.dump(RESULTS, open(f'{OUT}/results.json', 'w'), indent=1, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
np.savez(f'{OUT}/per_subject_lsd.npz', sids=np.array(DEV),
         **{f'{t}|{name}|{c}': subj(LSD[t][name][c]) for t in TARGETS for name in ORDER for c in COLS})

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)
show_rows = ['R1 GEO', PRIMARY, 'R2 GEO + REST', 'R4 GEO + REST + HEADM']
x = np.arange(len(COLS))
for ax, t in zip(axes, TARGETS):
    for i, name in enumerate(show_rows):
        ax.bar(x + (i - 1.5) * 0.2, [r2_of(PARTS[t][name][c]) for c in COLS], 0.2, label=name)
    ax.set_xticks(x); ax.set_xticklabels(COLS, fontsize=8); ax.axhline(0, color='k', lw=0.5)
    ax.set_title(f'{"measured" if t == "meas" else "simulated"} latent: CV-$R^2$ by band'); ax.set_ylabel('CV-$R^2$')
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/per_band.png', dpi=140); plt.show()
print(f'保存 -> {OUT}/results.json、per_subject_lsd.npz、per_band.png')


## 跑完之后

1. 把 Cell 7–10 的输出贴回来。
2. 把 `HRTF_Checkpoints/mesh2z_48k_rest_v1/` 里的 `results.json` 和 `per_band.png` 下载到 `hrtf-GitHub/mesh2z/output/mesh2z_48k_rest_v1/`（`per_subject_lsd.npz` 留在 Drive 或本地，不进仓库）。
3. PROJECT_LOG 记一条结果，按登记的判读规则写结论。
4. 论文：结果为零 → tab:ruled-out 加一行 "The rest of the head mesh"；结果为正 → 只能作为开发集上的探索性结果写进 4.2，不改 3.1% 的测试结果。
